In [28]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [29]:
PROJECT_ROOT = Path("..") ##masire project
DATA_DIR = PROJECT_ROOT / "data"
EVENTS_DIR = DATA_DIR / "events"
PLAYERS_DIR = DATA_DIR / "players"

FREIBURG_SQUAD_ID = 34

In [30]:
event_files = sorted(EVENTS_DIR.glob("events_*.json"))  ##peyda kardan file ha

print(f"Number of event files: {len(event_files)}")

Number of event files: 306


In [31]:
freiburg_files = [] ##chandta event freiburg darim vaghean

for file_path in event_files:
    with open(file_path, "r", encoding="utf-8") as file:
        events = json.load(file)

    squad_ids = {
        event.get("squadId")
        for event in events
        if event.get("squadId") is not None
    }

    if FREIBURG_SQUAD_ID in squad_ids:
        freiburg_files.append(file_path)

print(f"Freiburg event files: {len(freiburg_files)}")

Freiburg event files: 34


In [32]:
## JSON haro flatten mikonim be ye radif sade az data tabdil mikonim
def safe_nested_get(data, *keys, default=None):
    current = data

    for key in keys:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

        if current is None:
            return default

    return current

def extract_match_id(file_path: Path) -> int:  ## tabe estekhraje match ID
    return int(file_path.stem.split("_")[-1])

def flatten_event(event: dict, match_id: int) -> dict: ##tabe flatten eventha 
    return {
        "match_id": match_id,
        "event_id": event.get("id"),
        "event_index": event.get("index"),
        "period_id": event.get("periodId"),
        "time_seconds": safe_nested_get(
            event, "gameTime", "gameTimeInSec"
        ),
        "squad_id": event.get("squadId"),
        "player_id": safe_nested_get(event, "player", "id"),
        "player_position": safe_nested_get(
            event, "player", "position"
        ),
        "position_side": safe_nested_get(
            event, "player", "positionSide"
        ),
        "action_type": event.get("actionType"),
        "action": event.get("action"),
        "phase": event.get("phase"),
        "result": event.get("result"),
        "pressure": event.get("pressure"),
        "opponents": event.get("opponents"),
        "distance_to_goal": event.get("distanceToGoal"),
        "distance_to_opponent": event.get("distanceToOpponent"),
        "body_part": event.get("bodyPart"),
        "sequence_index": event.get("sequenceIndex"),
        "start_x": safe_nested_get(
            event, "start", "adjCoordinates", "x"
        ),
        "start_y": safe_nested_get(
            event, "start", "adjCoordinates", "y"
        ),
        "pitch_position": safe_nested_get(
            event, "start", "pitchPosition"
        ),
        "lane": safe_nested_get(event, "start", "lane"),
        "packing_zone": safe_nested_get(
            event, "start", "packingZone"
        ),
        "team_pxt": safe_nested_get(event, "pxT", "team"),
        "opponent_pxt": safe_nested_get(
            event, "pxT", "opponent"
        ),
    }

rows = [] ##hameye event haye freiburg

for file_path in freiburg_files:
    match_id = extract_match_id(file_path)

    with open(file_path, "r", encoding="utf-8") as file:
        events = json.load(file)

    for event in events:
        if event.get("squadId") != FREIBURG_SQUAD_ID:
            continue

        rows.append(flatten_event(event, match_id))

freiburg_events = pd.DataFrame(rows) ## sakhte ye dataframe ba event haye freiburg
#print(freiburg_events.info())
#print(freiburg_events.shape)
print(freiburg_events.head())


   match_id    event_id  event_index  period_id  time_seconds  squad_id  \
0    122841  4614770652           15          1       26.8640        34   
1    122841  4614770653           16          1       26.8641        34   
2    122841  4614770654           17          1       30.8100        34   
3    122841  4614770655           18          1       32.5430        34   
4    122841  4614770656           19          1       32.5431        34   

   player_id   player_position position_side action_type  ...  \
0        622  CENTRAL_DEFENDER  CENTRE_RIGHT   RECEPTION  ...   
1        622  CENTRAL_DEFENDER  CENTRE_RIGHT     DRIBBLE  ...   
2        622  CENTRAL_DEFENDER  CENTRE_RIGHT        PASS  ...   
3      23350  CENTRAL_DEFENDER   CENTRE_LEFT   RECEPTION  ...   
4      23350  CENTRAL_DEFENDER   CENTRE_LEFT     DRIBBLE  ...   

    distance_to_opponent  body_part sequence_index  start_x  start_y  \
0  MORE_THAN_FOUR_METERS  FOOT_HIGH              1     -3.5     16.7   
1  MORE_THAN_F

In [33]:
## مرحله ۴: اضافه کردن نام بازیکنان file players_743.json ro baz mikonam

with open(
    PLAYERS_DIR / "players_743.json",
    "r",
    encoding="utf-8",
) as file:
    raw_players = json.load(file)

##بعد یک DataFrame بساز. مثلاً اگر ساختار ساده باشد:

players = pd.DataFrame(
    [{"player_id": player.get("id"), "player_name": player.get("lastname"),} 
     for player in raw_players
    ]
)
"""""
print(freiburg_events['player_id'].is_unique)
print(players['player_id'].is_unique)
print(freiburg_events['player_id'].dtype)
print(players['player_id'].dtype)

players.info()
players.head()
players.shape

"""
## و merge:

freiburg_events = freiburg_events.merge(
    players,
    how="left",
    on="player_id",
)

print(players["player_name"])
## hala kole dataframe merge shode ba players haro baressi mikonim


0       Kramaric
1        Kimmich
2            Can
3           Dier
4          Weigl
         ...    
565        Chase
566    Mamutovic
567        Cissé
568       Norbye
569         Sato
Name: player_name, Length: 570, dtype: object


In [34]:
freiburg_events[
    ["player_id", "player_name"]
].drop_duplicates().head(30)

,player_id,player_name
0,622,Ginter
3,23350,Lienhart
6,49785,Atubolu
8,1320,Eggestein
14,1268,Günter
22,968,Gregoritsch
26,1294,Grifo
32,68933,Röhl
48,986,Kübler
60,862,Sallai


In [35]:
"""""
مرحله ۵: حذف Eventهای غیرقابل استفاده

بعضی Eventها بازیکن ندارند:

* سوت پایان
* خروج توپ
* Eventهای ویدیویی
* Eventهای سیستمی
"""

##برای player analytics:
player_events = freiburg_events.dropna( 
    subset=["player_id"]
).copy()

##ستون دقیقه:
player_events["minute"] = ( ##
    player_events["time_seconds"] / 60
)
## مختصات استاندارد:
player_events["plot_x"] = (
    player_events["start_x"] + 52.5
)

player_events["plot_y"] = (
    player_events["start_y"] + 34
)

##بعد خروجی را ذخیره کن:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

player_events.to_csv(
    OUTPUT_DIR / "freiburg_events.csv",
    index=False,
)